# Module 02 — Structured Output

**Goal:** Get reliable, parseable JSON from an LLM — not free-form text.

This notebook runs on the **Databricks free tier** against any OpenAI-compatible endpoint. Defaults to [GitHub Models](https://github.com/marketplace/models) (free).

By the end of this notebook you will know:
- Why free-form text output breaks production code
- Two ways to get JSON: prompt engineering vs JSON mode
- How Pydantic validates the structure after parsing
- How to defensively parse LLM output in the real world

---
**Run each cell in order (Shift+Enter).**

## Step 1 — Install dependencies

Install `openai` and `pydantic` once per session, then restart Python.

In [ ]:
%pip install --quiet openai pydantic
dbutils.library.restartPython()

## Step 2 — Set your parameters

Paste your [GitHub token](https://github.com/settings/tokens) into the `API_KEY` widget. Leave the other two at their defaults unless you want a different provider.

In [ ]:
dbutils.widgets.text("API_KEY", "", "API key")
dbutils.widgets.text("BASE_URL", "https://models.github.ai/inference", "Base URL")
dbutils.widgets.text("MODEL", "openai/gpt-4o-mini", "Model")

API_KEY = dbutils.widgets.get("API_KEY")
BASE_URL = dbutils.widgets.get("BASE_URL")
MODEL = dbutils.widgets.get("MODEL")

if not API_KEY:
    raise ValueError("Paste your API key into the API_KEY widget at the top of the notebook.")

from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"Model:    {MODEL}")
print(f"Endpoint: {BASE_URL}")

## The problem: free-form output is unparseable at scale

Ask the model to "generate SQL" with no format constraint and you get... whatever it feels like.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Write SQL to count rows in a table called 'users'."}
    ],
)

raw = response.choices[0].message.content
print(repr(raw))

The model might return:
- Plain SQL: `SELECT count(*) FROM users`
- SQL in a markdown fence: `` ```sql\nSELECT ...\n``` ``
- SQL with prose: `"Here's the query: SELECT count(*) FROM users"`
- A different phrasing every time

Writing a parser that handles all these variations is painful and brittle. The fix: **tell the model what format to use.**

## Approach 1 — Prompt engineering

Put the desired format in the system prompt. Works everywhere, but the model can still ignore it (especially smaller models).

In [ ]:
import json

SYSTEM = """
Always respond with a JSON object in exactly this format:
{"sql": "<your SELECT statement>", "explanation": "<one sentence>"}
Do not include any text outside the JSON object.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": "Count rows in a table called 'users'."},
    ],
    temperature=0,
)

raw = response.choices[0].message.content
print("Raw output:", raw)
print()
print("Parsed:", json.loads(raw))

## Approach 2 — JSON mode (`response_format`)

A stronger API-level guarantee: the model's output will always be valid JSON. You still need the system prompt to tell the model *which fields* to include.

> **Note:** GitHub Models, OpenAI, Groq, and recent Ollama builds all support `response_format`.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": "Count rows in a table called 'users'."},
    ],
    temperature=0,
    response_format={"type": "json_object"},
)

raw = response.choices[0].message.content
data = json.loads(raw)
print(data)
print(f"sql field:         {data['sql']}")
print(f"explanation field: {data['explanation']}")

## Step 3 — Validate with Pydantic

JSON is valid, but `json.loads()` still returns a plain `dict`. Pydantic turns it into a typed object and validates the fields.

In [ ]:
from pydantic import BaseModel, ValidationError

class SQLResponse(BaseModel):
    sql: str
    explanation: str


result = SQLResponse(**data)
print(f"Type: {type(result)}")
print(f"sql:  {result.sql}")
print(f"explanation: {result.explanation}")
print()

try:
    bad = SQLResponse(**{"sql": "SELECT 1"})
except ValidationError as e:
    print("Validation error caught:")
    print(e)

## Step 4 — Defensive parsing

Even with JSON mode, some models (especially smaller ones) wrap output in markdown fences like `` ```json ... ``` ``. Here's a defensive parser that handles common edge cases.

In [ ]:
import re

def parse_sql_response(raw: str) -> SQLResponse:
    text = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`").strip()

    try:
        data = json.loads(text)
        return SQLResponse(sql=data.get("sql", "").strip(),
                           explanation=data.get("explanation", "").strip())
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in LLM response:\n{raw}")

    data = json.loads(match.group())
    return SQLResponse(sql=data.get("sql", "").strip(),
                       explanation=data.get("explanation", "").strip())


test_cases = [
    '{"sql": "SELECT 1", "explanation": "Clean JSON"}',
    '```json\n{"sql": "SELECT 2", "explanation": "Fenced JSON"}\n```',
    'Here you go: {"sql": "SELECT 3", "explanation": "JSON in prose"}',
]

for case in test_cases:
    result = parse_sql_response(case)
    print(f"sql={result.sql!r:15}  explanation={result.explanation!r}")

## Step 5 — Putting it all together

Full flow: LLM call → raw text → defensive parse → validated Pydantic model.

In [ ]:
def generate_sql(question: str, schema_text: str) -> SQLResponse:
    system = """
You are a SQL assistant. Return only valid JSON with this exact structure:
{"sql": "<SELECT statement or empty string>", "explanation": "<one sentence>"}
Only generate SELECT statements. Never use DROP, DELETE, UPDATE, INSERT, etc.
"""
    user = f"Schema:\n{schema_text}\n\nQuestion: {question}"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return parse_sql_response(response.choices[0].message.content)


schema = "users: id (INTEGER), name (TEXT), email (TEXT), created_at (DATETIME)"
result = generate_sql("How many users signed up?", schema)

print(f"SQL:         {result.sql}")
print(f"Explanation: {result.explanation}")

## Summary

| Technique | Pros | Cons |
|-----------|------|------|
| Free-form output | No setup | Unparseable, inconsistent |
| Prompt engineering | Works everywhere | Model can ignore it |
| JSON mode | API-level guarantee | Not all providers support it |
| Pydantic validation | Type-safe, catches bad models | Requires defining schemas |

**In production, use all three together** (prompt + JSON mode + Pydantic).

**Next:** Module 03 — Tool Use. Instead of just generating text (or JSON), the LLM will *call functions* and decide what data to fetch.